# GA 2 — Dataset Assessment Practice

Use this notebook to solve the 7 questions on your own computer. Run the cells **in order** and inspect each output before entering your answers in the assessment.

**Dataset:** `GA_2_dataset.csv`

- In Google Colab: upload the CSV using the Files panel, or run the upload cell below.
- Locally: place the CSV in the same folder as this notebook.
- The notebook prints the result for each question; try predicting the result before running each answer cell.


In [ ]:
print(12)

: 

In [ ]:
import pandas as pd
import numpy as np

# If running in Google Colab, uncomment these two lines to upload the CSV:
# from google.colab import files
# files.upload()

# If your CSV is in another folder, change this path.
df = pd.read_csv("GA_2_dataset.csv")

print("Dataset shape:", df.shape)
display(df.head())
print("\nColumn data types:")
print(df.dtypes)


## Question 1 — Which columns have `object` datatype?

**Try first:** Which columns store text/categories rather than numeric values?

In [ ]:
# Q1: Display all columns whose dtype is object.
object_columns = df.select_dtypes(include=["object"]).columns.tolist()
print(object_columns)


## Question 2 — Male players from Europe with `InGamePurchases`

Count rows where `Gender` is Male, `Location` is Europe, and `InGamePurchases` indicates a purchase (`1`).

**Try first:** Write the three conditions in words.

In [ ]:
# Q2
q2_mask = (
    (df["Gender"] == "Male") &
    (df["Location"] == "Europe") &
    (df["InGamePurchases"] == 1)
)
print("Q2 answer:", int(q2_mask.sum()))


## Question 3 — Players under 18 who played strictly more than 10 hours

The conditions are `Age < 18` and `PlayTimeHours > 10`. “Strictly greater” means 10 itself does **not** count.

In [ ]:
# Q3
q3_mask = (df["Age"] < 18) & (df["PlayTimeHours"] > 10)
print("Q3 answer:", int(q3_mask.sum()))


## Question 4 — Total null values in the whole dataset

Count missing values in every column, then add those counts.

In [ ]:
# Q4
nulls_per_column = df.isna().sum()
print("Null values per column:")
print(nulls_per_column)
print("Q4 answer (total nulls):", int(nulls_per_column.sum()))


## Question 5 — Train/test split and least frequent class in `y_train`

- `EngagementLevel` is the target `y`.
- All other columns form `X`.
- Use `test_size=0.2` and `random_state=42`.
- Which class has the fewest values in `y_train`?

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["EngagementLevel"])
y = df["EngagementLevel"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training target counts:")
print(y_train.value_counts())
print("Q5 answer (least frequent class):", y_train.value_counts().idxmin())


## Shared preprocessing for Questions 6 and 7

Follow the instructions from the assessment:
- Calculate imputation statistics from the **training set only**.
- Exclude rows with missing values when calculating the statistics.
- Replace missing `Age` with the training mean.
- Replace missing `Location` with `"Other"`.
- Replace missing `GameDifficulty` with its most frequent training value.
- Replace missing `InGamePurchases` with `0`.

The helper below treats `NaN` as missing. Run this cell before Questions 6 and 7.

In [ ]:
# Work on copies so the original df remains unchanged.
X_train_imp = X_train.copy()
X_test_imp = X_test.copy()

# Use complete training rows (no missing values in any feature) to calculate
# the requested statistical values.
complete_train = X_train.dropna()

age_mean = complete_train["Age"].mean()
difficulty_mode = complete_train["GameDifficulty"].mode().iloc[0]

print("Training complete rows used for statistics:", len(complete_train))
print("Age mean:", age_mean)
print("GameDifficulty mode:", difficulty_mode)

# Apply the training-derived values to both train and test.
for part in (X_train_imp, X_test_imp):
    part["Age"] = part["Age"].fillna(age_mean)
    part["Location"] = part["Location"].fillna("Other")
    part["GameDifficulty"] = part["GameDifficulty"].fillna(difficulty_mode)
    part["InGamePurchases"] = part["InGamePurchases"].fillna(0)

print("\nMissing values remaining in train:", int(X_train_imp.isna().sum().sum()))
print("Missing values remaining in test:", int(X_test_imp.isna().sum().sum()))


## Question 6 — Sum of the imputed `Age` column in the test dataset

Use the imputed test features from the previous cell. Round the sum to 2 decimal places.

In [ ]:
# Q6
age_test_sum = X_test_imp["Age"].sum()
print("Q6 answer:", round(float(age_test_sum), 2))


## Question 7 — Preprocess features and sum the first five rows of transformed `X_test`

Steps:
1. Drop `PlayerID`.
2. Ordinally encode `GameDifficulty`: Easy=0, Medium=1, Hard=2.
3. One-hot encode `Gender`, `Location`, and `GameGenre`, using `drop_first=True`.
4. Standardize **all** transformed features using `StandardScaler`.
5. Fit encoding/scaling using training data only; transform the test data with those fitted choices.
6. Sum all values in the first five rows of transformed test features and round to 2 decimals.

The cell below uses the training columns to align the one-hot encoded test matrix, so test-only categories cannot create new feature columns.

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. Drop PlayerID before preprocessing.
Xtr = X_train_imp.drop(columns=["PlayerID"]).copy()
Xte = X_test_imp.drop(columns=["PlayerID"]).copy()

# 2. Ordinal encode GameDifficulty.
difficulty_map = {"Easy": 0, "Medium": 1, "Hard": 2}
Xtr["GameDifficulty"] = Xtr["GameDifficulty"].map(difficulty_map)
Xte["GameDifficulty"] = Xte["GameDifficulty"].map(difficulty_map)

# 3. One-hot encode nominal features; learn columns from training data.
nominal = ["Gender", "Location", "GameGenre"]
Xtr_enc = pd.get_dummies(Xtr, columns=nominal, drop_first=True, dtype=int)
Xte_enc = pd.get_dummies(Xte, columns=nominal, drop_first=True, dtype=int)

# Ensure test has exactly the same columns and order as train.
Xte_enc = Xte_enc.reindex(columns=Xtr_enc.columns, fill_value=0)

# Make sure all features are numeric.
Xtr_enc = Xtr_enc.astype(float)
Xte_enc = Xte_enc.astype(float)

# 4. Fit scaler on training data only; transform both datasets.
scaler = StandardScaler()
Xtr_scaled = scaler.fit_transform(Xtr_enc)
Xte_scaled = scaler.transform(Xte_enc)

# 5. Sum every value in the first five rows of transformed test features.
q7_sum = Xte_scaled[:5].sum()
print("Transformed test shape:", Xte_scaled.shape)
print("Q7 answer:", round(float(q7_sum), 2))


## Final check

Run this last cell to confirm that the target was kept separate from the feature matrix and the transformed test matrix contains no missing values.

In [ ]:
print("Target column:", y.name)
print("EngagementLevel in X?", "EngagementLevel" in X.columns)
print("PlayerID in transformed features?", "PlayerID" in Xtr_enc.columns)
print("Missing values in encoded train:", int(Xtr_enc.isna().sum().sum()))
print("Missing values in encoded test:", int(Xte_enc.isna().sum().sum()))
